# Solutions — S3 Semantics Locally: URIs, Prefixes, a boto3-like Client, Event-Driven Processing

Exercise notebook: `11_python_for_aws/01_s3_simulation.ipynb`. Each solution has: working code, an explanation, complexity, common mistakes, and a Java-style vs Pythonic comparison where it teaches something.

### Setup (same as the exercise notebook)

In [ ]:
from pathlib import Path
LAB = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datasets").is_dir() and (p / "labkit").is_dir())   # lab root (works from any folder)
S3_ROOT = LAB / "datasets/s3"        # local stand-in for S3: <root>/<bucket>/<key>
for p in sorted(S3_ROOT.rglob("*"))[:12]:
    print(p.relative_to(S3_ROOT))

## Exercise 1 — 🟢 Easy · Map S3 URIs to local paths safely

**Working solution**

In [ ]:
from pathlib import Path, PurePosixPath

def s3_to_local(uri, root):
    if not uri.startswith("s3://"):
        raise ValueError(f"not an s3 uri: {uri!r}")
    bucket, _, key = uri[5:].partition("/")
    if not bucket:
        raise ValueError("missing bucket")
    parts = PurePosixPath(key).parts if key else ()
    if ".." in parts:
        raise ValueError("path traversal is not allowed")
    return Path(root, bucket, *parts)

**Explanation**

`PurePosixPath` parses keys with `/` regardless of your OS; building the local path from parts works on Windows too. Never join untrusted keys blindly.

**Complexity:** O(len).

**Common mistakes**

- `Path(root) / key` with `key='../../etc/passwd'`.

## Exercise 2 — 🟡 Medium · List with prefix and delimiter

**Working solution**

In [ ]:
from pathlib import Path

def list_prefix(root, bucket, prefix="", delimiter=None):
    bucket_dir = Path(root) / bucket
    if not bucket_dir.is_dir():
        raise FileNotFoundError(f"no such bucket: {bucket}")
    contents, prefixes = [], {}
    for path in sorted(p for p in bucket_dir.rglob("*") if p.is_file()):
        key = path.relative_to(bucket_dir).as_posix()
        if not key.startswith(prefix):
            continue
        rest = key[len(prefix):]
        if delimiter and delimiter in rest:
            common = prefix + rest.split(delimiter, 1)[0] + delimiter
            prefixes[common] = None
            continue
        contents.append({"Key": key, "Size": path.stat().st_size})
    return {"Contents": contents, "CommonPrefixes": [{"Prefix": p} for p in prefixes]}

**Explanation**

S3 has no directories; the console's "folders" are `CommonPrefixes` computed from a delimiter. Understanding this explains why `ls`-style listing of a huge prefix is expensive and why
partition layouts (`key=value/`) matter so much for Athena/Glue/Spark partition pruning.

**Complexity:** O(objects in bucket) here; O(objects under prefix) in real S3.

**Common mistakes**

- Treating prefix `'sales'` as a folder — it also matches `'sales_archive/...'`.

## Exercise 3 — 🟡 Medium · 'Rename' a prefix (copy + delete)

**Working solution**

In [ ]:
import shutil
from pathlib import Path

def _local(uri, root):
    bucket, _, key = uri[5:].partition("/")
    return Path(root, bucket, *[p for p in key.split("/") if p])

def move_prefix(root, src_uri, dst_uri, delete_source=True):
    src_dir, dst_dir = _local(src_uri, root), _local(dst_uri, root)
    if dst_dir == src_dir or src_dir in dst_dir.parents:
        raise ValueError("destination must not be inside source")
    files = sorted(p for p in src_dir.rglob("*") if p.is_file())
    for f in files:
        target = dst_dir / f.relative_to(src_dir)
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(f, target)
    if delete_source:
        for f in files:
            f.unlink()
        for d in sorted((p for p in src_dir.rglob("*") if p.is_dir()), key=lambda p: len(p.parts), reverse=True):
            if not any(d.iterdir()):
                d.rmdir()
        if src_dir.exists() and not any(src_dir.iterdir()):
            src_dir.rmdir()
    return len(files)

**Explanation**

Copy-all-then-delete keeps the data safe if a copy fails midway (you'd rerun; copies are idempotent). On real S3 this is `copy_object` per key — O(objects) API calls, which is why you design layouts so you never need to 'rename'.

**Complexity:** O(objects + bytes).

**Common mistakes**

- Deleting each source right after copying it — a failure leaves data split across two places.

## Exercise 4 — 🟠 Hard · A boto3-compatible local S3 client

**Working solution**

In [ ]:
import hashlib
import io
from pathlib import Path

class ClientError(Exception):
    def __init__(self, code, message, operation):
        super().__init__(f"An error occurred ({code}) when calling the {operation} operation: {message}")
        self.response = {"Error": {"Code": code, "Message": message}}
        self.operation_name = operation

class LocalS3Client:
    def __init__(self, root):
        self.root = Path(root)

    def _bucket(self, bucket, op):
        path = self.root / bucket
        if not path.is_dir():
            raise ClientError("NoSuchBucket", f"The specified bucket does not exist: {bucket}", op)
        return path

    def _object(self, bucket, key, op):
        path = self._bucket(bucket, op).joinpath(*key.split("/"))
        if not path.is_file():
            raise ClientError("NoSuchKey", f"The specified key does not exist: {key}", op)
        return path

    @staticmethod
    def _etag(data):
        return f'"{hashlib.md5(data).hexdigest()}"'

    def create_bucket(self, Bucket, **_):
        (self.root / Bucket).mkdir(parents=True, exist_ok=True)
        return {"Location": f"/{Bucket}"}

    def put_object(self, Bucket, Key, Body=b"", **_):
        data = Body.encode("utf-8") if isinstance(Body, str) else bytes(Body)
        path = self._bucket(Bucket, "PutObject").joinpath(*Key.split("/"))
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(data)
        return {"ETag": self._etag(data)}

    def head_object(self, Bucket, Key, **_):
        data = self._object(Bucket, Key, "HeadObject").read_bytes()
        return {"ContentLength": len(data), "ETag": self._etag(data)}

    def get_object(self, Bucket, Key, **_):
        data = self._object(Bucket, Key, "GetObject").read_bytes()
        return {"Body": io.BytesIO(data), "ContentLength": len(data), "ETag": self._etag(data)}

    def delete_object(self, Bucket, Key, **_):
        path = self._bucket(Bucket, "DeleteObject").joinpath(*Key.split("/"))
        path.unlink(missing_ok=True)
        return {}

    def list_objects_v2(self, Bucket, Prefix="", MaxKeys=1000, ContinuationToken=None, **_):
        bucket_dir = self._bucket(Bucket, "ListObjectsV2")
        keys = sorted(p.relative_to(bucket_dir).as_posix() for p in bucket_dir.rglob("*") if p.is_file())
        keys = [k for k in keys if k.startswith(Prefix) and (ContinuationToken is None or k > ContinuationToken)]
        page, truncated = keys[:MaxKeys], len(keys) > MaxKeys
        response = {"KeyCount": len(page), "IsTruncated": truncated, "MaxKeys": MaxKeys, "Prefix": Prefix}
        if page:
            response["Contents"] = [{"Key": k, "Size": (bucket_dir / k).stat().st_size, "ETag": self._etag((bucket_dir / k).read_bytes())} for k in page]
        if truncated:
            response["NextContinuationToken"] = page[-1]
        return response

**Explanation**

Matching boto3's interface (CamelCase kwargs, response dicts, `ClientError.response['Error']['Code']`) means business code written against this fake runs unchanged against AWS —
the same idea as `moto`. Note that missing `Contents` on empty listings is a real boto3 quirk that breaks naive `response["Contents"]` code.

**Complexity:** O(objects) per list here.

**Common mistakes**

- `response['Contents']` on an empty prefix → `KeyError` in production.
- Catching `Exception` instead of checking the error code.

## Challenge — 🔴 Very Hard · An S3-event-driven, idempotent handler (Lambda style)

**Working solution**

In [ ]:
import csv
import hashlib
import io
import json
from pathlib import Path
from urllib.parse import unquote_plus

VALID_STATUS = {"PLACED", "SHIPPED", "DELIVERED", "CANCELLED", "RETURNED"}
OUT_BUCKET = "lab-processed-bucket"

def _validate(row):
    if not (row.get("order_id") or "").strip():
        raise ValueError("missing order_id")
    if int(row["quantity"]) < 1:
        raise ValueError("quantity < 1")
    if float(row["unit_price"]) < 0:
        raise ValueError("negative price")
    if (row.get("status") or "").strip().upper() not in VALID_STATUS:
        raise ValueError(f"bad status {row.get('status')!r}")
    return {**row, "status": row["status"].strip().upper(), "quantity": int(row["quantity"]), "unit_price": float(row["unit_price"])}

def _write_jsonl(path, rows):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text("".join(json.dumps(r, sort_keys=True) + "\n" for r in rows), encoding="utf-8")

def handle_s3_event(event, root):
    root = Path(root)
    result = {"processed": [], "skipped": [], "ignored": 0, "errors": [], "valid_rows": 0, "rejected_rows": 0}
    for record in event.get("Records", []):
        bucket = record["s3"]["bucket"]["name"]
        key = unquote_plus(record["s3"]["object"]["key"])
        if not record.get("eventName", "").startswith("ObjectCreated") or not key.endswith(".csv"):
            result["ignored"] += 1
            continue
        source = root.joinpath(bucket, *key.split("/"))
        try:
            data = source.read_bytes()
        except FileNotFoundError as exc:
            result["errors"].append({"key": key, "error": f"{type(exc).__name__}: {exc}"})
            continue
        md5 = hashlib.md5(data).hexdigest()
        marker = root.joinpath(OUT_BUCKET, "_markers", bucket, *key.split("/")).with_name(f"{Path(key).name}.{md5}.done")
        if marker.exists():
            result["skipped"].append(key)
            continue
        valid, rejected = [], []
        for row in csv.DictReader(io.StringIO(data.decode("utf-8-sig"))):
            try:
                valid.append(_validate(row))
            except (ValueError, KeyError, TypeError) as exc:
                rejected.append({**row, "_error": str(exc)})
        out_key = key[:-4] + ".jsonl"
        _write_jsonl(root.joinpath(OUT_BUCKET, "clean", *out_key.split("/")), valid)
        if rejected:
            _write_jsonl(root.joinpath(OUT_BUCKET, "rejected", *out_key.split("/")), rejected)
        marker.parent.mkdir(parents=True, exist_ok=True)
        marker.write_text("", encoding="utf-8")
        result["processed"].append(key)
        result["valid_rows"] += len(valid)
        result["rejected_rows"] += len(rejected)
    return result

**Explanation**

S3 notifications are **at-least-once**: the same event can arrive twice, and a retry after a timeout re-sends the whole batch. Content-addressed markers (key + MD5) make the
handler idempotent, and writing the marker *last* means a crash mid-way simply reprocesses. Per-record error capture keeps one missing object from failing the whole batch.

**Complexity:** O(total object bytes).

**Common mistakes**

- Forgetting to URL-decode keys (`ingest_date%3D2026` ≠ `ingest_date=2026`).
- Marker written before outputs → data loss on crash.